 # 11.1 梯度消失与梯度爆炸的问题

理解**梯度消失（Vanishing Gradients）**和**梯度爆炸（Exploding Gradients）**

在第 10 章中，我们使用的是只有 2-3 层的浅层网络，这些问题并不明显。但在第 11 章，当我们试图构建 10 层、20 层甚至上百层网络时，这些问题会直接导致模型无法训练。

---

### 1. 问题的根源：链式法则的连乘效应

神经网络的学习依赖于**反向传播（Backpropagation）**。其核心数学工具是**链式法则**。

想象梯度（误差信号）从输出层出发，一层一层向输入层反向传播。在每一层，梯度都会乘以该层的权重 $w$。
$$\text{Gradient at Layer } L \approx \text{Gradient at Output} \times w_L \times w_{L-1} \times \dots \times w_1$$



#### A. 梯度消失 (Vanishing Gradients)
如果网络权重的初始值较小（例如 $|w| < 1$），或者使用了 Sigmoid 这种会将输出压缩到 $(0, 1)$ 之间的激活函数：
* **现象**：当层数很多时，这些小于 1 的项不断连乘，梯度会呈**指数级缩小**。
* **后果**：传到靠近输入层（底层）时，梯度几乎变成了 0。底层神经元的权重永远得不到更新，模型无法学习。

#### B. 梯度爆炸 (Exploding Gradients)
如果权重的初始值较大（例如 $|w| > 1$）：
* **现象**：梯度在连乘过程中呈**指数级增长**。
* **后果**：梯度变得巨大，导致权重更新步长过大。
* **表现**：在 PyCharm 控制台中，会看到 **Loss 突然变成 `NaN`**，或者模型完全崩溃。

---

### 2. 问题重要的原因

书中提到了两个导致问题的“罪魁祸首”：

1.  **激活函数选择不当**：
    经典的 **Sigmoid** 激活函数在输入很大或很小时，导数接近 0（饱和区）。这会加速梯度的消失。
2.  **权重初始化不当**：
    如果所有权重都使用标准正态分布随机初始化，每一层输出的方差会随着层数增加而增大，最终导致信号饱和。

---


## 11.1.1 Glorot和He初始化


---
### 一、核心背景：为什么需要权重初始化？
深度神经网络训练的核心痛点是**不稳定梯度问题**：
- 正向预测时，信号需要正常流动；反向传播梯度时，信号方向相反。
- 若初始化不当，会出现两种极端：
  1.  **梯度消失**：信号逐层衰减，深层网络无法有效学习；
  2.  **梯度爆炸**：信号逐层放大，模型训练发散。
- 理想状态：**每层输出的方差 = 输入的方差**，且反向传播的梯度方差也保持一致，让信号在网络中稳定流动。

---
### 二、Glorot（Xavier）初始化
#### 1. 提出者与核心思想
由 Glorot 和 Bengio 在论文中提出，也叫 **Xavier 初始化**（以第一作者命名），核心是折中方案：
- 无法同时保证「正向输出方差一致」和「反向梯度方差一致」（除非输入/神经元数量相等），因此取两者的平均值来平衡。

#### 2. 关键定义
- `fan_in`：该层的**输入神经元数量**（输入维度）
- `fan_out`：该层的**输出神经元数量**（该层神经元数）
- `fan_avg = (fan_in + fan_out) / 2`：输入/输出维度的平均值

#### 3. 公式（逻辑激活函数场景）
##### 正态分布初始化
权重服从均值为0的正态分布，方差为：
$$
\sigma^2 = \frac{1}{fan_{avg}}
$$

##### 均匀分布初始化
权重在 $[-r, +r]$ 区间内均匀分布，其中：
$$
r = \sqrt{\frac{3}{fan_{avg}}}
$$

#### 4. 适用场景
- 适配 **sigmoid、tanh、softmax** 这类饱和激活函数；
- Keras 默认使用均匀分布的 Glorot 初始化。

---
### 三、LeCun 初始化
#### 1. 提出者与核心
由 Yann LeCun 在20世纪90年代提出，是 Glorot 初始化的变体：
- 将 Glorot 公式中的 `fan_avg` 替换为 `fan_in`，仅以输入维度为基准。

#### 2. 核心特点
- 当 `fan_in = fan_out` 时，LeCun 初始化等价于 Glorot 初始化；
- 适配 **SELU 激活函数**（推荐配合正态分布使用）。

---
### 四、He 初始化（针对ReLU系列）
#### 1. 提出者与核心
由 Kaiming He 等人在2015年论文中提出，专门针对 **ReLU 及其变体（ELU等）** 这类非饱和激活函数设计，解决ReLU场景下Glorot初始化的方差失衡问题。

#### 2. 核心参数（对应表11-1）
| 初始化方法 | 适配激活函数       | 正态分布方差 $\sigma^2$ |
|------------|--------------------|--------------------------|
| Glorot     | tanh/logistic/softmax | $1/fan_{avg}$            |
| He         | ReLU和变体         | $2/fan_{in}$             |
| LeCun      | SELU               | $1/fan_{in}$             |

#### 3. Keras 中的使用
- 可通过 `kernel_initializer="he_normal"`（正态分布）或 `kernel_initializer="he_uniform"`（均匀分布）指定；
- 若需基于 `fan_avg` 而非 `fan_in` 的He初始化，可使用 `VarianceScaling` 自定义：
  ```python
  he_avg_init = keras.initializers.VarianceScaling(
      scale=2., mode='fan_avg', distribution='uniform'
  )
  keras.layers.Dense(10, activation="sigmoid", kernel_initializer=he_avg_init)
  ```

---
### 五、关键总结
1.  **初始化的本质**：通过控制权重的方差，保证网络中信号（正向）和梯度（反向）的稳定流动，避免消失/爆炸，加速训练收敛，是深度学习成功的核心技巧之一。
2.  **选型原则**：
    - 用sigmoid/tanh/softmax → 选 **Glorot 初始化**
    - 用ReLU/LeakyReLU/ELU → 选 **He 初始化**
    - 用SELU → 选 **LeCun 初始化**
3.  **补充类比（注2）**：类似麦克风放大器链，每个放大器（网络层）的增益（权重初始化）必须适中：太弱则信号消失，太强则信号饱和失真，只有正确设置才能让声音（信号）全程清晰。

---


## 11.1.2 非饱和激活函数

传统的 Sigmoid 和 Tanh 函数在输入值较大或较小时，导数会趋近于 0（即进入“饱和区”），导致梯度消失。为了打破这个僵局，研究者们开发了一系列在正值区域（甚至负值区域）保持梯度不为 0 的激活函数。

---

### 1. ReLU (Rectified Linear Unit)
这是目前的深度学习标准配置。它的公式非常简单：$ReLU(z) = \max(0, z)$。

* **优点**：计算速度极快；在正值区域永远不会饱和。
* **致命伤：神经元坏死 (Dying ReLUs)**。
    * **现象**：由于负半区导数恒为 0，如果一个神经元的权重更新导致其输入始终为负，那么这个神经元将永远输出 0，且梯度永远为 0。
    * **结果**：这个神经元就“死掉了”，不再对训练产生任何贡献。

---

### 2. LeakyReLU (Leaky ReLU)
为了解决神经元坏死问题，LeakyReLU 给负半区留了一个小小的“缝隙”：$LeakyReLU_\alpha(z) = \max(\alpha z, z)$。

* **参数 $\alpha$**：通常设为 $0.01$。这个微小的斜率确保了即使输入为负，神经元依然有微弱的梯度传回，从而有机会“复活”。
* **RReLU (Randomized LeakyReLU)**：在训练期间随机选择 $\alpha$，在测试期间固定。这能起到一定的正则化作用。

---

### 3. ELU (Exponential Linear Unit)
ELU 是书中非常推崇的一个变体，它的性能往往优于所有的 ReLU 变体。
$$ELU_\alpha(z) = \begin{cases} \alpha(e^z - 1) & z < 0 \\ z & z \ge 0 \end{cases}$$

* **优点**：
    1.  **均值接近零**：负半区的指数曲线让输出的均值更接近 0，这有助于加快收敛速度。
    2.  **处处可导**：在 $z=0$ 处也是平滑的（如果 $\alpha=1$），减少了震荡。
* **缺点**：涉及指数运算，计算速度比 ReLU 慢。但在训练时，收敛速度的提升通常能弥补计算成本。

---

### 4. SELU (Scaled ELU)
这是 ELU 的缩放版本。作者提到，如果构建一个**纯全连接层（Dense）**的深层网络，且满足以下条件：
1.  输入特征已标准化。
2.  权重使用 **LeCun 初始化**。
3.  网络架构是顺序的（Sequential）。

**那么模型将实现“自归一化（Self-Normalizing）”**。这意味着每一层的输出都会自动保持均值为 0、方差为 1，从而彻底解决梯度消失和爆炸。


---

### 总结：该选哪一个？

书中给出的建议顺序（优先级从高到低）：
1.  **SELU**：如果网络是纯全连接层且能满足自归一化条件，它是首选。
2.  **ELU**：通用的高性能选择。
3.  **LeakyReLU**：如果你在意计算速度。
4.  **ReLU**：最基础的选择（由于其简单性，依然是很多人的首选）。
5.  **Sigmoid/Tanh**：除非是在输出层（如二分类），否则**绝不**在隐藏层使用。

In [2]:
import warnings
# 屏蔽所有 UserWarning 类型的警告
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
# 如何在keras中实现
from tensorflow import keras

model = keras.models.Sequential([
    # 1. 直接使用字符串调用 ReLU 或 ELU
    keras.layers.Dense(100, activation="elu", kernel_initializer="he_normal"),

    # 2. 使用 LeakyReLU (作为独立层，放在 Dense 之后)
    keras.layers.Dense(100, kernel_initializer="he_normal"),
    keras.layers.LeakyReLU(alpha=0.2),

    # 3. 使用 SELU (必须配合 LeCun 初始化)
    keras.layers.Dense(100, activation="selu", kernel_initializer="lecun_normal")
])

## 11.1.3 批量归一化

**批量归一化（Batch Normalization，简称 BN）** 是由 Sergey Ioffe 和 Christian Szegedy 在 2015 年提出的一项革命性技术。
它能极大地加速深层网络的收敛，甚至允许你使用不那么完美的初始化方法。

---

### 1. 核心原理：解决“内部协变量偏移”

在深层网络中，每一层参数的更新都会改变下一层输入的分布。随着层数加深，这种微小的改变会不断累积，导致底层网络陷入饱和区，这就是**内部协变量偏移 (Internal Covariate Shift)**。

**BN 的做法很简单粗暴：** 在每一层的激活函数**之前**（或之后），强行将该批次（Batch）的数据拉回到均值为 0、方差为 1 的标准分布，然后再进行缩放和偏移。

---

### 2. 数学步骤（针对每个特征）

对于一个 Batch 的输入 $B = \{x_1, \dots, x_m\}$，BN 执行以下四步：

1.  **计算均值**：$\mu_B = \frac{1}{m} \sum_{i=1}^m x_i$
2.  **计算方差**：$\sigma_B^2 = \frac{1}{m} \sum_{i=1}^m (x_i - \mu_B)^2$
3.  **标准化**：$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$ （$\epsilon$ 是为了防止除以 0）
4.  **缩放与偏移**：$z_i = \gamma \hat{x}_i + \beta$

**注意**：$\gamma$（缩放）和 $\beta$（偏移）是模型**学习出来**的参数。这意味着如果模型发现“标准正态分布”并不适合这一层，它可以通过这两个参数把数据还原回去。

---

### 3. 为什么 BN 这么强大？

* **缓解梯度消失**：通过将输入限制在非饱和区（尤其是对于 Sigmoid/Tanh），梯度能更顺畅地传导。
* **允许更大的学习率**：由于每一层的分布稳定，可以放心地把 `learning_rate` 调大，从而成倍缩短训练时间。
* **轻微的正则化作用**：BN 在计算均值和方差时引入了 Batch 的随机性（噪声），这起到了类似 Dropout 的效果，能减少过拟合。
* **对初始化不敏感**：有了 BN，即使没用 He 初始化，模型通常也能跑通。


---

### 注意事项

1.  **预测时的表现**：在测试（预测）单条数据时，并没有“Batch”的概念。Keras 会自动使用训练期间计算的**移动平均值**来代替。
2.  **计算开销**：BN 会给每一层增加额外的计算量，虽然它能减少 Epoch 总数，但每个 Epoch 的运行时间会稍微变长一点。

### 用keras实现批量归一化

In [6]:
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(300, activation="selu", kernel_initializer="he_normal"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(100, activation="elu", kernel_initializer="he_normal"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(10,activation='softmax')
])

In [7]:
# 显示模型摘要
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 784)            │         3,136 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 300)            │       235,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 300)            │         1,200 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 271,346 (1.04 MB)

 Trainable params: 268,978 (1.03 MB)

 Non-trainable params: 2,368 (9.25 KB)

In [8]:
# 查看BN层的参数
[(var.name,var.trainable) for var in model.layers[1].variables]

[('gamma', True),
 ('beta', True),
 ('moving_mean', False),
 ('moving_variance', False)]

In [10]:
# 删除偏置项
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(300, kernel_initializer="he_normal",use_bias=False),
    keras.layers.Activation("elu"),
    keras.layers.Dense(100, kernel_initializer="he_normal",use_bias=False),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('elu'),
    keras.layers.Dense(10,activation='softmax')
])

## 11.1.4 梯度裁剪

如果在训练深层网络（尤其是循环神经网络 RNN，或者没有使用批量归一化的网络）时，发现 Loss 突然变成 `NaN`，或者控制台报错显示梯度过大，那么**梯度裁剪（Gradient Clipping）**就是解决方案

在第 11 章中，梯度裁剪是应对**梯度爆炸**最直接、最有效的手段。

---

### 1. 核心原理：

梯度爆炸的本质是反向传播时梯度变得极其巨大，导致权重更新步长过大，模型直接飞出了“最优解区域”。

**梯度裁剪的做法**：在优化器更新权重**之前**，先检查梯度的大小。如果梯度的模（Norm）超过了设定的阈值，就强行把它缩小到这个阈值范围内。

* **形象比喻**：就像在下山，如果前方是个悬崖（梯度巨大），梯度裁剪会强行让只迈出一小步，而不是直接跳下去，从而保证依然留在山坡上寻找路径。

---

### 2. Keras 中的两种裁剪方式

在 PyCharm 中配置优化器时，可以通过两个参数来实现裁剪：`clipnorm` 和 `clipvalue`。

* **优点**：由于方向没变，模型依然沿着预期的最快下降方向移动，只是步子变小了。这在大多数情况下比 `clipvalue` 更稳健。

---

### 总结：
1.  **合适的初始化**：He 初始化或 Glorot 初始化（预防）。
2.  **批量归一化**：实时调整分布（抑制）。
3.  **梯度裁剪**：最后的保险丝（兜底）。


In [11]:
# 按值裁剪,将所有梯度的分量限制在-1.0到1.0之间
optimizer = keras.optimizers.SGD(clipvalue = 1.0)

* **缺点**：它可能会改变梯度的**方向**。例如，如果原始梯度向量是 $[0.9, 100.0]$，裁剪后变成 $[0.9, 1.0]$。可以看到，原本几乎垂直向上的方向被强行偏转了。

# 11.2 重用预训练层

**重用预训练层（Reusing Pretrained Layers）**，也就是常说的**迁移学习（Transfer Learning）**，是深度学习中最具性价比的技巧。

它的核心逻辑非常直观：**在面临一个业务时不必从头开始训练模型，可以将已经训练好的和本业务类似的模型（预训练模型）迁移过来使用**

---

### 1. 为什么要重用？

* **节省数据**：深度神经网络通常需要海量数据。如果样本很少（比如只有几百张图），从头训练必然过拟合。而预训练模型已经从数百万张图中学到了通用的特征。
* **节省算力**：训练一个大型模型可能需要数周时间，重用模型只需在数据上微调几小时甚至几分钟。
* **性能更强**：通常情况下，迁移学习得到的模型比从零训练的模型准确率更高。

---

### 2. 核心操作：冻结与解冻

在重用模型时，不能直接开始训练，因为随机初始化的新层会产生巨大的梯度，这会把预训练层里辛苦学到的权重直接“冲坏”。

**标准的流程如下：**

1.  **加载模型**：加载一个在类似任务上训练好的模型A。
2.  **创建模型 B**：去掉模型 A 的输出层，换上适合任务的新输出层。
3.  **冻结 (Freezing)**：将模型 A 的所有旧层设为不可训练（`trainable = False`）。此时，训练只会更新新加的那一层。
4.  **训练新层**：运行几个 Epoch，直到新层初步收敛。
5.  **解冻与微调 (Unfreezing & Fine-Tuning)**：解冻模型 A 的顶层（或者全部），以极低的学习率再次训练。

---

### 3. 什么时候该用迁移学习？

书中给出了一个非常有用的判断标准：

| 数据量    | 与原任务的相似度 | 策略 |
|:-------| :--- | :--- |
| **很少** | **很高** | 仅训练输出层。 |
| **较多** | **很高** | 训练输出层 + 微调顶层。 |
| **很少** | **很低** | 效果可能不佳，尝试只用最底下的 1-2 层。 |
| **很多** | **很低** | 可以从头训练，或者用预训练模型作为更好的初始化点。 |

---


## 11.2.1 用keras进行迁移学习

In [12]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

# 1. 加载 Fashion MNIST 原始数据集
(x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.fashion_mnist.load_data()

# 2. 数据归一化（像素值缩放到 0~1）
x_train_full = x_train_full / 255.0
x_test_full = x_test_full / 255.0

# 3. 筛选出 8 个类别（排除标签 5=凉鞋、6=衬衫）
# 保留的类别：0(T恤),1(裤子),2(套头衫),3(连衣裙),4(外套),7(运动鞋),8(包),9(短靴)
train_mask = ~np.isin(y_train_full, [5, 6])
test_mask = ~np.isin(y_test_full, [5, 6])

x_train = x_train_full[train_mask]
y_train = y_train_full[train_mask]
x_test = x_test_full[test_mask]
y_test = y_test_full[test_mask]

# 4. 标签重映射（原标签不连续，重映射为 0~7，方便分类）
# 原标签: [0,1,2,3,4,7,8,9] → 新标签: [0,1,2,3,4,5,6,7]
label_map = {0:0, 1:1, 2:2, 3:3, 4:4, 7:5, 8:6, 9:7}
y_train = np.array([label_map[y] for y in y_train])
y_test = np.array([label_map[y] for y in y_test])

# 5. 增加通道维度（适配CNN输入：(28,28) → (28,28,1)）
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

print(f"训练集形状: {x_train.shape}, 标签数: {len(np.unique(y_train))}")
print(f"测试集形状: {x_test.shape}")


训练集形状: (48000, 28, 28, 1), 标签数: 8
测试集形状: (8000, 28, 28, 1)


In [13]:
# 构建模型A（8分类）
model_A = keras.models.Sequential([
    # 卷积层1
    keras.layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(28,28,1)),
    keras.layers.MaxPooling2D((2,2)),
    # 卷积层2
    keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    keras.layers.MaxPooling2D((2,2)),
    # 卷积层3
    keras.layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    keras.layers.MaxPooling2D((2,2)),
    # 全连接层
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    # 输出层：8个类别，softmax激活
    keras.layers.Dense(8, activation='softmax')
])

# 编译模型
model_A.compile(
    loss='sparse_categorical_crossentropy',  # 标签为整数，用此损失
    optimizer='adam',
    metrics=['accuracy']
)

# 查看模型结构
model_A.summary()

# 训练模型
history = model_A.fit(
    x_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1  # 用10%训练集做验证
)

# 在测试集上评估
test_loss, test_acc = model_A.evaluate(x_test, y_test)
print(f"\n模型A测试集准确率: {test_acc:.4f}")

# 保存模型（对应书中的 my_model_A.h5）
model_A.save("my_model_A.h5")
print("模型A已保存为 my_model_A.h5")

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 128)            │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 241,288 (942.53 KB)

 Trainable params: 241,288 (942.53 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step - accuracy: 0.8821 - loss: 0.3386 - val_accuracy: 0.9260 - val_loss: 0.2145
Epoch 2/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.9311 - loss: 0.1961 - val_accuracy: 0.9392 - val_loss: 0.1751
Epoch 3/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.9424 - loss: 0.1628 - val_accuracy: 0.9465 - val_loss: 0.1551
Epoch 4/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.9503 - loss: 0.1417 - val_accuracy: 0.9458 - val_loss: 0.1482
Epoch 5/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.9556 - loss: 0.1236 - val_accuracy: 0.9475 - val_loss: 0.1484
Epoch 6/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.9617 - loss: 0.1080 - val_accuracy: 0.9508 - val_loss: 0.1418
Epoch 7/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.9660 - loss: 0.0950 - val_accuracy: 0.9475 - val_loss: 0.1487
Epoch 8/15
675/675 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.9700 - loss: 0.0827 - val


模型A测试集准确率: 0.9514
模型A已保存为 my_model_A.h5


In [14]:
# 加载模型A，并基于该模型的层创建一个新模型，重用除输出层之外的所有层
model_A = keras.models.load_model("my_model_A.h5")
model_B_on_A = keras.models.Sequential(model_A.layers[:-1])
model_B_on_A.add(keras.layers.Dense(1,activation='sigmoid'))

In [15]:
# 克隆模型A的架构并复制其权重
model_A_clone = keras.models.clone_model(model_A)
model_A_clone.set_weights(model_A.get_weights())

In [16]:
# 新的输出层是随机初始化的，会产生较大的错误，存在较大的错误梯度，会破坏重用的权重，可以在前几轮次时冻结重用的层，给新层一些时间来学习合理的权重，也就是将每一层的可训练属性设置为False并编译模型
for layer in model_B_on_A.layers[:-1]:
    layer.trainable = False

model_B_on_A.compile(loss = 'binary_crossentropy',
                     optimizer = 'sgd',
                     metrics = ['accuracy'])


In [17]:
# 从原始 8 分类数据中筛选出两个类别的样本
class_a = 0   # T恤
class_b = 7   # 运动鞋（原标签映射后的 5，因为 label_map 中 7→5）
# 注意： y_train 已经是重映射后的 0~7，其中：
# 原7(运动鞋) -> 5, 原8(包)->6, 原9(短靴)->7
# 要区分 T恤(0) 和 运动鞋(5)
mask_train = (y_train == 0) | (y_train == 5)
mask_test = (y_test == 0) | (y_test == 5)

X_train_B = x_train[mask_train]
y_train_B = (y_train[mask_train] == 5).astype(int)   # 运动鞋为1，T恤为0

X_test_B = x_test[mask_test]
y_test_B = (y_test[mask_test] == 5).astype(int)

print(X_train_B.shape, y_train_B.shape)  # 检查

(12000, 28, 28, 1) (12000,)


In [18]:
# 解冻重用的层之后，需要再次编译模型，可以选择降低学习率避免损坏重用的权重
history = model_B_on_A.fit(X_train_B,
                           y_train_B,
                           epochs = 4,
                           validation_data = (X_test_B, y_test_B))

for layer in model_B_on_A.layers[:-1]:
    layer.trainable = True

optimizer = keras.optimizers.SGD(learning_rate=1e-4)
model_B_on_A.compile(
    loss = 'binary_crossentropy',
    optimizer = optimizer,
    metrics = ['accuracy']
)
history = model_B_on_A.fit(X_train_B,
                           y_train_B,
                           validation_data = (X_test_B, y_test_B))

Epoch 1/4
375/375 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9997 - loss: 0.0153 - val_accuracy: 0.9975 - val_loss: 0.0100
Epoch 2/4
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9998 - loss: 0.0045 - val_accuracy: 0.9980 - val_loss: 0.0071
Epoch 3/4
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9998 - loss: 0.0030 - val_accuracy: 0.9985 - val_loss: 0.0058
Epoch 4/4
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9998 - loss: 0.0023 - val_accuracy: 0.9985 - val_loss: 0.0051
375/375 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9998 - loss: 0.0020 - val_accuracy: 0.9985 - val_loss: 0.0051


In [19]:
# 评估模型
model_B_on_A.evaluate(X_test_B, y_test_B)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9985 - loss: 0.0051


[0.00505820894613862, 0.9984999895095825]

## 11.2.2 无监督预训练

**无监督预训练（Unsupervised Pretraining）** 是一个非常具有启发性的概念。它解决了深度学习中一个最现实的痛点：**如果手头有海量的数据，但其中只有极少一部分是有标签的，该怎么办？**

---

### 1. 核心逻辑：

无监督预训练的本质是：先让模型在**无标签数据**上进行“自学习”，掌握数据的内在结构和特征，然后再在**有标签数据**上进行微调。

**它的步骤通常如下：**
1.  **找一个辅助任务**：在没有标签的情况下，强迫模型去完成一个任务（例如：把模糊的图片变清晰，或者预测图片中缺失的部分）。
2.  **训练特征提取器**：通过完成这个辅助任务，模型的前几层会自动学会识别线条、纹理、形状等通用特征。
3.  **迁移与微调**：重用这些预训练好的层，在其上方添加一个新的输出层，用珍贵的“有标签数据”进行最后的微调。

---

### 2. 经典实现方案：自动编码器（Autoencoders）

虽然第 17 章会详细讲自动编码器，但第 11 章提到了它是实现无监督预训练的主力军。

* **栈式自动编码器（Stacked Autoencoders）**：
    * 通过一层层地堆叠，每一层都试图重构前一层的输入。
    * 为了重构输入，模型必须抓住数据中最本质的特征。
* **GAN（生成对抗网络）**：
    * 通过让生成器和判别器相互博弈，生成器也能学到极强的特征表示能力。

---

### 3. 什么时候需要它？

当满足以下三个条件时，可以考虑无监督预训练：
1.  **数据极度不平衡**：无标签数据量 $\gg$ 有标签数据量。
2.  **任务极其复杂**：从零开始训练很难收敛。
3.  **没有类似的预训练模型**：如果你找不到别人训练好的模型（比如你的数据是深海声呐图像），你就得自己通过无监督预训练来“造”一个底座。

---


## 11.2.3 辅助任务的预训练

核心逻辑是：**如果没有目标任务的标签，就人为创造一个不需要人工标注、但能强迫模型理解数据特征的“假任务”。**

---

### 1. 核心思想：自我监督（Self-Supervision）

辅助任务通常利用数据自身的属性来生成标签。这种方法在计算机视觉（CV）和自然语言处理（NLP）领域几乎是统治级的存在。

* **例子 A（图像旋转）**：将图片随机旋转 0°、90°、180°、270°，让模型预测旋转的角度。
    * *原理*：模型为了猜对角度，必须先理解图片里什么是“头”、什么是“地”，从而学会了物体识别的基础特征。
* **例子 B（拼图游戏）**：把一张图片切成 9 块并打乱顺序，让模型预测它们的正确位置。
    * *原理*：模型必须理解物体的局部与整体关系才能拼对。
* **例子 C（颜色恢复）**：把彩色图片变黑白，让模型预测每个像素点的颜色。

---

### 2. 执行流程：从“热身”到“正赛”

在 PyCharm 中实现辅助任务预训练，通常遵循以下三个阶段：

1.  **阶段一：辅助任务训练**
    * 准备海量无标签数据。
    * 自动生成辅助标签（如：自动旋转图片并记录角度）。
    * 训练模型 A 解决这个辅助任务。
2.  **阶段二：知识迁移**
    * 保留模型 A 的底层（特征提取层）。
    * 丢弃辅助任务的输出层（如：那个预测 4 个角度的分类层）。
3.  **阶段三：目标任务微调**
    * 在模型 A 的上方添加目标任务的输出层（如：预测房价或识别肿瘤）。
    * 用少量珍贵的有标签数据进行训练。

---

### 3.

虽然两者都属于无监督/自监督学习，但辅助任务在现代深度学习中更受青睐：

* **更深层次的语义**：自动编码器有时会走捷径，只学会了简单的像素级拷贝；而辅助任务（如拼图）强迫模型理解物体的结构语义。
* **计算效率**：有些辅助任务（如对比学习）在超大规模数据集上的收敛效果比重构整个图像更好。


---


### 总结：

* **迁移学习**：重用别人在类似任务上练好的模型。
* **无监督预训练**：用自动编码器自我重构。
* **辅助任务预训练**：用“假任务”强迫模型理解数据结构。

# 11.3 更好的优化器


以下是动量优化的核心机制与实战要点：

---

### 1. 核心思想：利用惯性 (Inertia)

在标准 SGD 中，每一步更新只取决于**当前的梯度**。如果当前坡度平缓，更新就慢；如果两边坡度陡峭但中间窄（如“峡谷”地形），SGD 会在两壁之间剧烈震荡，缓慢前行。

**动量优化引入了“动量向量” $m$**：
* 它不仅考虑当前的梯度，还会记住**之前的运动方向和速度**。
* **物理直观**：随着小球向下滚动，重力（梯度）会不断给它加速度，使其越来越快，直到达到终端速度（受摩擦力或动量衰减系数限制）。

---

### 2. 数学原理（两步更新法）

动量优化通过增加一个超参数 $\beta$（动量衰减系数）来控制“惯性”的大小：

1.  **更新动量向量**：
    $$m \leftarrow \beta m + \eta \nabla_\theta J(\theta)$$
    *(这里 $\beta$ 模拟了摩擦力，防止动量无限增加；$\eta$ 是学习率)*
2.  **更新权重**：
    $$\theta \leftarrow \theta - m$$

**结果**：
* 在梯度方向一致的区域，动量会不断累积，速度越来越快（加速）。
* 在梯度反复震荡的区域（如峡谷），正负梯度会相互抵消，从而**平滑震荡**。

---

### 3. 超参数 $\beta$ 的设置

* **常用值**：通常设为 **0.9**。
* **物理含义**：$\beta = 0.9$ 意味着当前的动量由 90% 的旧动量和 10% 的新梯度组成。
* **影响**：
    * $\beta=0$：等同于标准 SGD。
    * $\beta$ 越大：惯性越强，能更快冲过平原，但也可能在到达最优解时“冲过头”，需要更多时间减速停止。

---

### 4. 为什么它比 SGD 快

1.  **冲破平原**：在 Loss 曲线极其平坦的区域（梯度接近 0），SGD 几乎停滞，但动量优化能靠惯性快速滑过。
2.  **抑制震荡**：在凹槽地形中，它能抵消垂直于前进方向的力，让模型更专注于向目标中心移动。


---


In [20]:
# Keras 实战

#在 PyCharm 中，不需要手动写公式，只需在 `SGD` 优化器中配置 `momentum` 参数：

from tensorflow import keras

# 基础动量优化
optimizer = keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)

# 应用到模型
model.compile(loss="mse", optimizer=optimizer)


## 11.3.2 Nesterov加速梯度

**Nesterov 加速梯度（Nesterov Accelerated Gradient，简称 NAG）** 是对普通动量优化（Momentum）的一个高效的改进。

如果说**动量优化**是让小球盲目地顺着坡度加速滚下，那么 **NAG** 就是让小球在每一步滚动之前，先往前方“探探路”。

---

### 1. 核心逻辑：预判未来 (Look-ahead)

在标准的动量优化中，梯度是在**当前位置** $\theta$ 计算的。
但在 NAG 中，由于我们知道动量下一步肯定会将模型推向某个方向，那我们干脆**先跳到那个预期的位置**，在那儿测量梯度。

* **普通动量**：计算当前坡度 $\rightarrow$ 加上之前的惯性 $\rightarrow$ 更新。
* **NAG**：顺着惯性先往前看一眼 $\rightarrow$ 计算那里的坡度 $\rightarrow$ 结合当前惯性进行修正更新。

---

### 2. 为什么 NAG 更优秀？

想象正在滑雪：
* **普通动量**：只根据脚下的坡度加速。当发现快要撞到山谷对面的斜坡时，可能因为速度太快而停不下来，产生剧烈震荡。
* **NAG**：会观察前方。当预见到前方坡度开始上升时（梯度变号），会提前减速。

**结论**：NAG 能够提供一个“阻尼”效果，使模型在接近最优解时更加稳定，减少冲过头（Overshoot）的现象，收敛速度通常比普通动量快得多。

---

### 3. 数学公式（简版）

1.  **预估下一步位置**：$\theta + \beta m$
2.  **在预估位置计算梯度**：$\nabla_\theta J(\theta + \beta m)$
3.  **更新动量与权重**：
    $$m \leftarrow \beta m + \eta \nabla_\theta J(\theta + \beta m)$$
    $$\theta \leftarrow \theta - m$$

---

### 5. 什么时候使用 NAG？

* 当决定使用 **带动量的随机梯度下降（SGD + Momentum）** 时，**永远应该开启 `nesterov=True`**。它几乎没有任何额外的计算开销，但性能提升非常稳健。
* 在第 11 章中，如果发现 `Adam` 优化器让模型在验证集上不稳定，切换回 `SGD + NAG` 往往能获得更好的泛化效果。

---

### 知识点串联
到目前为止，已经看过了优化器进化的前两个阶段：
1.  **SGD**：老实走路。
2.  **Momentum**：带惯性走路。
3.  **NAG**：带惯性且看路。


In [21]:


# Keras 实战

#在 PyCharm 中，不需要手动实现这个预判逻辑。NAG 并没有独立的类，它是作为 `SGD` 优化器的一个可选参数存在的。

from tensorflow import keras

# 只需要将 nesterov 设为 True
optimizer = keras.optimizers.SGD(learning_rate=0.001,
                                 momentum=0.9,
                                 nesterov=True)

model.compile(loss="mse", optimizer=optimizer)

## 11.3.3 AdaGrad

**AdaGrad** 是理解后续所有自适应优化器的基石。

---

### 1. 核心思想：因材施教 (Adaptive)

在之前的 SGD、Momentum 或 NAG 中，所有参数（权重和偏置）都共用同一个学习率 $\eta$。
**AdaGrad 的直觉是**：
* 如果一个参数的梯度很大（说明更新很频繁），我们就调小它的学习率。
* 如果一个参数的梯度很小（说明更新很稀疏），我们就保持较大的学习率。

**物理类比**：就像在崎岖的山路上行走，对于坡度非常陡的地方，我们小碎步挪动（减小学习率）；对于平缓的地方，我们大跨步前进（保持学习率）。

---

### 2. 数学原理：累积梯度的平方

AdaGrad 通过一个累加变量 $s$ 来记录每个参数自训练开始以来所有梯度的平方和：

1.  **累积平方梯度**：
    $$s \leftarrow s + \nabla_\theta J(\theta) \otimes \nabla_\theta J(\theta)$$
    *(这里 $\otimes$ 表示逐元素乘法。$s$ 向量记录了每个参数梯度的“历史总量”。)*

2.  **更新参数**：
    $$\theta \leftarrow \theta - \eta \nabla_\theta J(\theta) \oslash \sqrt{s + \epsilon}$$
    *($\eta$ 是初始学习率，$\epsilon$ 是防止除以零的微小值。)*

**关键点**：由于 $s$ 是不断累加的正数，分母 $\sqrt{s + \epsilon}$ 会越来越大，导致实际的学习率逐渐变小。

---

### 3. AdaGrad 的优缺点

####  优点：
* **自动调节**：减少了手动调节学习率的需求。
* **处理稀疏数据**：对于那些不经常更新的参数（稀疏特征），它能维持足够的步长，这在处理文本数据（NLP）时非常有用。

#### 致命缺点（在深度网络中）：
* **早衰（Premature Convergence）**：由于 $s$ 是**单调递增**的，学习率会**永远递减**。在深度神经网络中，往往还没等模型到达最优解，学习率就已经减小到几乎为零，最终模型无法更新。
* **这就是为什么书上说：** AdaGrad 在简单的线性回归中表现不错，但**绝不要**用它来训练深度神经网络。

---


## 11.3.4 RMSProp

**RMSProp** 是由 Hinton 在其 Coursera 课程中提出的一种优化算法。它是为了解决 **AdaGrad** 学习率下降过快、导致模型提前停止学习的问题而诞生的。

---

### 1. 核心改进：指数衰减平均 (Exponential Decay)

AdaGrad 失败的原因在于它累积了**自训练开始以来**所有的梯度平方。随着时间推移，分母变得巨大，步长几乎消失。

**RMSProp 的解决方法**：它不再平权地累积所有历史梯度，而是使用**指数衰减移动平均**。这意味着它会更看重“最近”几个 Batch 的梯度，而逐渐忘记很久以前的梯度。


---

### 2. 数学公式

RMSProp 引入了一个新的超参数 $\rho$（衰减率，通常设为 0.9）：

1.  **更新梯度平方的移动平均**：
    $$s \leftarrow \rho s + (1 - \rho) \nabla_\theta J(\theta) \otimes \nabla_\theta J(\theta)$$
    *($s$ 不再是无止境增加的累加和，而是一个动态调整的平均值)*
2.  **更新参数**：
    $$\theta \leftarrow \theta - \eta \nabla_\theta J(\theta) \oslash \sqrt{s + \epsilon}$$

---

### 3. 为什么 RMSProp 更好用？

* **适应性强**：它依然保留了为每个参数定制学习率的能力（梯度大的参数步长小，梯度小的参数步长大）。
* **不会早衰**：由于使用了衰减机制，分母 $s$ 不会无限制增长。即使训练了成千上万次，学习率依然能保持在有效范围内。
* **收敛迅速**：在很多复杂的非凸优化问题（如深度神经网络）中，RMSProp 的表现通常远好于 AdaGrad 和简单的动量优化。

---


In [22]:
#  Keras 实战

# 在 PyCharm 中，除非在处理非常特殊的循环神经网络（RNN），否则你通常不需要调整 rho。

from tensorflow import keras

# rho 默认就是 0.9，这在绝大多数任务中都是最优的
optimizer = keras.optimizers.RMSprop(learning_rate=0.001, rho=0.9)

model.compile(loss="mse", optimizer=optimizer)

## 11.3.5 Adam和Nadam优化


---

## 一、核心概念总览
Adam（Adaptive Moment Estimation，自适应矩估计）是**当前深度学习最主流的优化器**，它融合了**动量优化（Momentum）**和**RMSProp**的核心思想，是自适应学习率算法的集大成者。

### 核心原理
- 同时跟踪两个指数衰减平均值：
  1.  **一阶矩（动量项）**：过去梯度的指数衰减平均值（对应动量优化）
  2.  **二阶矩（平方梯度项）**：过去平方梯度的指数衰减平均值（对应RMSProp）
- 对两个矩进行**偏差修正**，解决训练初期矩值偏向0的问题
- 用修正后的矩自动调整每个参数的学习率，实现自适应更新

---

## 二、Adam 算法完整公式
> 注：$t$ 为迭代次数（从1开始），$\otimes$ 为元素-wise乘法，$\sqrt{\cdot}$ 为元素-wise开方

| 步骤 | 公式 | 含义 |
|------|------|------|
| 1 | $\boldsymbol{m} \leftarrow \beta_1 \boldsymbol{m} - (1-\beta_1) \nabla_\theta J(\theta)$ | 更新一阶矩（动量）：保留历史动量 $\beta_1 \boldsymbol{m}$，叠加当前梯度 |
| 2 | $\boldsymbol{s} \leftarrow \beta_2 \boldsymbol{s} + (1-\beta_2) \nabla_\theta J(\theta) \otimes \nabla_\theta J(\theta)$ | 更新二阶矩（平方梯度）：保留历史平方梯度 $\beta_2 \boldsymbol{s}$，叠加当前梯度平方 |
| 3 | $\hat{\boldsymbol{m}} \leftarrow \frac{\boldsymbol{m}}{1-\beta_1^t}$ | 一阶矩偏差修正：抵消训练初期矩值偏小的问题 |
| 4 | $\hat{\boldsymbol{s}} \leftarrow \frac{\boldsymbol{s}}{1-\beta_2^t}$ | 二阶矩偏差修正：同上 |
| 5 | $\theta \leftarrow \theta - \eta \hat{\boldsymbol{m}} \oslash \sqrt{\hat{\boldsymbol{s}} + \varepsilon}$ | 更新参数：用修正后的矩做自适应更新，$\oslash$ 为元素-wise除法 |

---

## 三、关键超参数与默认值
| 超参数 | 含义 | 标准默认值 | 说明 |
|--------|------|------------|------|
| $\beta_1$ | 一阶矩（动量）衰减率 | 0.9 | 控制历史梯度的权重，默认值效果极佳 |
| $\beta_2$ | 二阶矩（平方梯度）衰减率 | 0.999 | 控制历史平方梯度的权重，默认值效果极佳 |
| $\eta$（lr） | 全局学习率 | 0.001 | Adam对学习率不敏感，默认值通常无需调整 |
| $\varepsilon$ | 数值稳定项 | $10^{-7}$ | 防止分母为0，Keras默认使用 `keras.backend.epsilon()` |



## 四、Adam 与其他优化器的核心区别
| 优化器 | 核心特点 | 与Adam的关系 |
|--------|----------|--------------|
| 普通动量（Momentum） | 仅跟踪梯度的指数衰减平均值，无自适应学习率 | Adam的一阶矩部分继承了动量的思想 |
| RMSProp | 仅跟踪平方梯度的指数衰减平均值，无动量项 | Adam的二阶矩部分继承了RMSProp的思想 |
| AdaGrad | 累加所有历史梯度平方，学习率持续衰减，易过早收敛 | RMSProp/Adam用指数衰减替代累加，解决了过早收敛问题 |
| Adam | 同时跟踪一阶+二阶矩+偏差修正，自适应学习率 | 融合动量+RMSProp，是当前最通用的优化器 |

---

## 五、Adam 的变体算法
### 1. AdaMax
- **核心改进**：将Adam中二阶矩的 $\ell_2$ 范数（平方和开根号）替换为 $\ell_\infty$ 范数（时间衰减梯度的最大值）
- **公式简化**：删除Adam的第4步（二阶矩偏差修正），用 $\max(\beta_2 s, |\nabla_\theta J(\theta)|)$ 替代平方梯度累加
- **特点**：理论上更稳定，但实际效果通常略逊于Adam，仅作为Adam效果不佳时的备选

### 2. Nadam（Nesterov-accelerated Adam）
- **核心改进**：在Adam的基础上，融入 Nesterov加速梯度（NAG）的“前瞻”思想
- **特点**：收敛速度通常比Adam更快，整体表现优于RMSProp，但部分场景下泛化性略差
- **适用场景**：需要更快收敛速度的任务，作为Adam的进阶替代

---

## 六、关键注意事项与实践准则
### 1. 自适应优化器的潜在风险
Ashia C. Wilson 等人2017年的研究表明：**Adam等自适应优化器可能在某些数据集上泛化性不佳**
- 解决方案：若模型验证集效果差，可尝试改用普通Nesterov加速梯度（带动量的SGD）
- 补充：自适应优化器收敛速度远快于SGD，是训练深度网络的首选

### 2. 优化器选择实用建议
- **首选**：Adam（默认参数即可，无需大量调参，适配绝大多数深度学习任务）
- **备选**：
  - 若Adam泛化性差 → 换带动量的SGD/NAG
  - 若需要更快收敛 → 换Nadam
  - 若Adam效果不稳定 → 尝试AdaMax
- **二阶优化器**：基于Hessian矩阵的算法因计算量过大，不适合深度神经网络（参数动辄百万级，无法存储/计算Hessian）

---
## 七、延伸：稀疏模型训练
所有上述优化器生成的都是**密集模型**（绝大多数参数非零），若需要低内存、高推理速度的模型：
1.  **简单方法**：训练后将绝对值很小的权重置零（但会降低模型性能，且稀疏度有限）
2.  **更好方法**：训练时加入 $\ell_1$ 正则化，主动诱导权重稀疏（在本章后续内容详细介绍）

---

In [24]:

# Adam 优化器标准调用
optimizer = keras.optimizers.Adam(
    learning_rate=0.001,        # 学习率 η
    beta_1=0.9,      # 一阶矩衰减率 β₁
    beta_2=0.999,    # 二阶矩衰减率 β₂
    epsilon=1e-7     # 稳定项 ε
)


### 优化器比较
| 类别 | 收敛速度 | 收敛质量 |
|:--- |:--- |:--- |
| **SGD** | * | *** |
| **SGD(momentum=...)** | ** | *** |
| **SGD(momentum=..., nesterov=True)** | ** | *** |
| **Adagrad** | *** | *（停止太早） |
| **RMSprop** | *** | ** 或者 *** |
| **Adam** | *** | ** 或者 *** |
| **Nadam** | *** | ** 或者 *** |
| **AdaMax** | *** | ** 或者 *** |

> **注**：表格中星级符号含义：*（差）、**（平均）、***（好）。

## 11.3.6 学习率调度

一个固定的学习率通常很难奏效：太大则无法收敛（在最优解附近震荡），太小则训练极慢。

书中介绍了几种最实用的调度策略：

---

### 1. 为什么要“调度”？
理想的训练过程应该是：
* **初期**：步子迈大一点，快速离开随机初始化的混乱区域。
* **后期**：步子迈小一点，精细地落入全局最优解的谷底。

---

### 2. 常见的调度策略

#### A. 幂调度 (Power Scheduling)
学习率根据迭代次数 $t$ 稳步下降：$\eta(t) = \eta_0 / (1 + t/s)^c$。
* **特点**：初期下降快，后期下降慢。
* **Keras 实现**：在优化器中直接设置 `decay` 参数（在新版 Keras 中更推荐使用 `LearningRateSchedule` 对象）。

#### B. 指数调度 (Exponential Scheduling)
学习率每隔一定的步数就减少为原来的十分之一：$\eta(t) = \eta_0 \cdot 0.1^{t/s}$。
* **特点**：这是最常用的策略之一，下降非常平滑。


#### C. 分段恒定调度 (Piecewise Constant Scheduling)
手动指定在哪些 Epoch 使用哪个学习率。例如：前 5 个 Epoch 用 0.1，之后用 0.01。
* **缺点**：需要非常了解模型，手动调参工作量大。

#### D. 性能调度 (Performance Scheduling / ReduceLROnPlateau)
**这是书中最实用的建议**。每当验证集上的 Loss 停止下降（进入平台期）时，就将学习率乘以一个因子（如 0.5）。

#### E. 1周期调度 (1cycle Scheduling)
由 Leslie Smith 提出。它先在训练前半段将学习率线性增加到极大值，后半段再线性减小。
* **优势**：能显著加速训练，并起到正则化作用，防止过拟合。

---

### 3. Keras 中的两种实现方式
可以通过两种方式实现调度：

1.  **Callbacks（回调函数）**：在每个 Epoch 结束时更新学习率。
    * *优点*：易于读取验证集 Loss（如 `ReduceLROnPlateau`）。
    * *用法*：`model.fit(..., callbacks=[lr_scheduler])`。
2.  **Scheduler Objects（调度对象）**：在每个迭代步（Step）更新学习率。
    * *用法*：直接传给优化器 `optimizer = keras.optimizers.SGD(learning_rate=lr_schedule)`。

---

### 总结：

书中给出的实战建议顺序：
1.  **首选**：**1周期调度**（1cycle），它的提速效果最明显。
2.  **次选**：**指数调度** 或 **性能调度（ReduceLROnPlateau）**。它们非常稳健且易于实现。
3.  **避坑**：不要迷信默认的固定学习率，哪怕只是简单的指数衰减也能让模型表现提升一大截。

---

### 提示
运行 fit 时，可以配合 `TensorBoard` 观察学习率的变化曲线。如果看到 Loss 曲线在某一点突然大幅下降，通常就是学习率调度开始发力了。



In [25]:
# 幂调度
optimizer = keras.optimizers.SGD(learning_rate=0.01,decay = 1e-4)

In [26]:
# 指数调度
def exponential_decay_fn(epoch):
    return 0.01 * 0.1**(epoch/20)

# 或者
def exponential_decay(lr0,s):
    def exponential_decay_fn_1(epoch):
        return exponential_decay_fn_1

exponential_decay_fn = exponential_decay(lr0 = 0.01,s = 20)

In [30]:
#创建一个LearningRateScheduler回调函数
lr_scheduler = keras.callbacks.LearningRateScheduler(exponential_decay_fn)

In [31]:
# 分段恒定函数
def piecewise_constant_fn(epoch):
    if epoch<5:
        return 0.01
    elif epoch<15:
        return 0.005
    else:
        return 0.001

In [32]:
# 性能调度
# 如果 validation loss 5个epoch没提升，学习率乘以 0.5
lr_scheduler = keras.callbacks.ReduceLROnPlateau(factor=0.5,patience=5)

In [33]:
# 1周期回调
s = 20*len(x_train) // 32
learning_rate = keras.optimizers.schedules.ExponentialDecay(0.01,s,0.1)
optimizer = keras.optimizers.SGD(learning_rate=learning_rate)

# 11.4 通过正则化避免过拟合


权重衰减（Weight Decay）是防止模型过拟合的经典手段。

---

### 1. $l_1$ 和 $l_2$ 正则化的核心逻辑

深度学习模型之所以过拟合，通常是因为某些权值 $w$ 变得异常大，导致模型对训练数据的细微波动过度敏感。正则化的本质是在 **损失函数（Loss）** 中加入一个惩罚项：
$$\text{Total Loss} = \text{Original Loss} + \text{Regularization Term}$$

#### A. $l_2$ 正则化（权重衰减）
* **做法**：惩罚项是所有权重平方的和 $\frac{1}{2} \sum w^2$。
* **效果**：它会让权重向 0 靠拢，但不会真正等于 0。它倾向于让权重分布得更加**均匀且微小**。
* **直观理解**：迫使模型利用所有的输入特征，而不是依赖于某几个特别大的特征。

#### B. $l_1$ 正则化（稀疏化）
* **做法**：惩罚项是所有权重绝对值的和 $\sum |w|$。
* **效果**：它会产生**稀疏权重**，即让很多不重要的权重直接变成 **0**。
* **直观理解**：它在做正则化的同时，顺便帮你做了“特征选择”，剔除掉那些无关紧要的输入。

---

### 总结与对比

| 特性 | $l_1$ 正则化 | $l_2$ 正则化 |
| :--- | :--- | :--- |
| **惩罚项** | 权重的绝对值之和 | 权重的平方和 |
| **对权重的影响** | 使权重趋向于 **0** (稀疏) | 使权重趋向于 **小数值** |
| **主要用途** | 特征选择、模型压缩 | 防止过拟合、提升泛化性能 |
| **推荐程度** | 除非需要稀疏性，否则少用 | **深度学习的首选** |


In [34]:
from functools import partial

RegularizedDense = partial(keras.layers.Dense,
                           activation='elu',
                           kernel_initializer='he_uniform',
                           kernel_regularizer=keras.regularizers.l2(0.01))

model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28,28]),
    RegularizedDense(300),
    RegularizedDense(100),
    RegularizedDense(10,activation='softmax',kernel_initializer='glorot_uniform')
])

## 11.4.2 dropout

在第 11 章中，**Dropout** 被称为最有效的正则化技术之一。它由 Geoffrey Hinton 团队提出，并在 AlexNet 夺冠后迅速走红。

---

### 1. 核心机制：随机失活

Dropout 的操作非常简单且极具破坏性：在每个训练步（Training Step）中，每个神经元（包括输入层，但不包括输出层）都有一个概率 $p$ 被暂时“关掉”。

* **训练阶段**：被选中的神经元在该步迭代中输出为 0，不参与前向传播，也不参与反向传播。
* **测试阶段**：所有神经元都保持激活状态。

---

### 2. 为什么 Dropout 有效？

1.  **防止“共适应”（Co-adaptation）**：
    由于一个神经元不能依赖于相邻的神经元（因为邻居随时可能被关掉），它被迫学会**独立**提取有用的特征。这使得模型更加稳健。
2.  **集成学习（Ensemble Learning）**：
    每次迭代时，实际上都在训练一个不同的、更小的子网络。因为训练了成千上万次，最终的模型可以看作是无数个微型子网络的“平均值”。这种集成效应极大增强了泛化能力。

---

### 3. 一个关键的技术细节：缩放 (Scaling)

由于测试时所有神经元都参与工作，而训练时只有一部分，这会导致测试时的总输入信号远大于训练时。

* **解决办法**：在训练结束后，将权重乘以 $(1-p)$，或者（现在更常用的）在训练时将存活的神经元输出除以 $(1-p)$。Keras 会自动帮你完成这一切，不需要操心。

---

### 4. 书中的实战建议

* **丢弃率 $rate$ 的选择**：通常设为 **0.2 到 0.5**。
    * 较小的网络：0.1 - 0.2。
    * 较大的网络或过拟合严重：0.5。
* **模型观察**：如果发现模型在训练集上表现远好于验证集（严重过拟合），就增加 Dropout 率；如果模型在训练集上都学不动（欠拟合），就降低 Dropout 率。
* **特殊情况**：
    * **SELU**：如果在使用自归一化网络（SELU），应使用专门的 `AlphaDropout`，它能保持数据的均值和方差。
    * **BN 层**：虽然 BN 也有一定的正则化作用，但很多顶级架构会同时使用 BN 和 Dropout。


In [35]:

# 在 PyCharm 中，只需像添加普通层一样添加 `Dropout` 层。通常放在 `Dense` 层之后：

from tensorflow import keras

model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dropout(rate=0.2), # 输入层的 Dropout
    keras.layers.Dense(300, activation="elu", kernel_initializer="he_normal"),
    keras.layers.Dropout(rate=0.2), # 隐藏层的 Dropout
    keras.layers.Dense(100, activation="elu", kernel_initializer="he_normal"),
    keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(10, activation="softmax")
])

## 11.4.3 蒙特卡罗（MC）Dropout


### 1. 核心逻辑：从“确定”到“概率”

普通的 Dropout 在测试（预测）时是关闭的，模型对同一个输入永远给出同一个固定的输出。
**MC Dropout 的做法是**：在测试时也**开启** Dropout，并对同一个样本进行多次预测。

* **原理**：由于 Dropout 是随机丢弃神经元，每次预测时面对的其实是原始模型的一个微小“变体”（子网络）。
* **操作**：对同一个输入运行 $N$ 次（例如 100 次）前向传播。
* **结果**：得到了一组预测值，通过计算这组值的**平均值**和**标准差**，就得到了更稳健的结果

---

### 2. 为什么要用 MC Dropout？

1.  **提升准确率**：通常情况下，MC Dropout 的平均预测结果比单次预测更精确，因为它利用了集成学习的原理。
2.  **不确定性估计 (Uncertainty Estimation)**：
    * 如果 100 次预测结果高度一致（标准差小），说明模型很自信。
    * 如果 100 次预测结果天差地别（标准差大），说明模型在“瞎猜”。这在医疗诊断、自动驾驶等容错率极低的领域至关重要。
3.  **升级**：它不需要改变模型架构，也不需要重新训练，直接拿现成的模型就能用。

---

### 3. 什么时候该用它？

* **风险评估**：如果需要知道模型什么时候在“不懂装懂”。
* **性能压榨**：当已经用尽了所有手段，想再提升 0.5% 的准确率。
* **注意**：由于要预测多次（如 100 次），它的推理速度会变慢 100 倍。在对实时性要求极高的场景（如高频交易）中需要权衡。

---


In [36]:
# 运行100次预测，并强制开启训练模式（即开启Dropout）
y_probas = np.stack([model(x_test_full,training = True) for sample in range(100)])
# 计算100次预测的平均值
y_proba = y_probas.mean(axis = 0)

In [37]:
np.round(model.predict(x_test_full), 2)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


array([[0.23, 0.04, 0.08, ..., 0.08, 0.07, 0.1 ],
       [0.2 , 0.02, 0.02, ..., 0.02, 0.06, 0.04],
       [0.12, 0.05, 0.15, ..., 0.14, 0.08, 0.03],
       ...,
       [0.2 , 0.04, 0.07, ..., 0.12, 0.07, 0.02],
       [0.07, 0.04, 0.2 , ..., 0.17, 0.05, 0.05],
       [0.17, 0.07, 0.06, ..., 0.09, 0.07, 0.11]],
      shape=(10000, 10), dtype=float32)

In [38]:
np.round(y_proba[:1,2])

array([0.], dtype=float32)

In [39]:
y_std = y_probas.std(axis = 0)
np.round(y_std[:1], 2)

array([[0.09, 0.02, 0.06, 0.05, 0.09, 0.05, 0.06, 0.04, 0.05, 0.07]],
      dtype=float32)

## 11.4.4 最大范数正则化


---

### 1. 核心原理：给权重设“天花板”

对于每个神经元，最大范数正则化会限制其输入连接权重的 $l_2$ 范数，使其满足：
$$\|w\|_2 \le r$$
其中 $r$ 是设定的**最大范数超参数**。

**它是如何工作的？**
在每次训练迭代之后，模型会检查每个神经元的权重向量 $w$：
* 如果 $\|w\|_2 \le r$，则保持不变。
* 如果 $\|w\|_2 > r$，则将权重按比例缩小：$w \leftarrow w \frac{r}{\|w\|_2}$。

---

### 2. 为什么它很强大？

1.  **防止梯度爆炸**：由于权重被硬性限制在一个范围内，即使学习率设得很高，权重也不会无止境地膨胀，从而有效避免了梯度爆炸。
2.  **不影响 Loss 函数**：它不在损失函数中添加惩罚项，因此不会直接干扰梯度的计算方向。它只是在更新后做一次“修剪”。
3.  **配合其它技术**：它是 **Dropout** 的完美搭档。Dropout 有时会导致权重变得很大以补偿失活的神经元，而最大范数正则化能完美抑制这种副作用。

---

### 4. $l_2$ vs 最大范数正则化：该选哪个？

| 特性 | $l_2$ 正则化 | 最大范数正则化 |
| :--- | :--- | :--- |
| **实现方式** | 在 Loss 函数中添加惩罚项 | 在更新后对权重进行缩放（约束） |
| **权重倾向** | 倾向于让权重变小且均匀 | 允许权重在上限内自由浮动 |
| **学习率敏感度** | 学习率过大会导致 $l_2$ 失效 | **对高学习率有极强的免疫力** |
| **推荐组合** | 基础配置首选 | **配合 Dropout 或不稳定网络时首选** |

---

### 第 11 章：正则化技术的最终选择策略

书中对这一章的所有正则化给出了一个清晰的**优先级指南**：

1.  **默认方案**：**早停法 (Early Stopping)**。这是最省钱、最有效的正则化。
2.  **通用增强**：**$l_2$ 正则化**。适用于大多数网络。
3.  **深层/复杂网络**：**Dropout**。目前工业界最稳健的防过拟合手段。
4.  **遇到梯度问题时**：**最大范数正则化**。尤其是在你希望使用较大的学习率来加速收敛时。

---


In [40]:
keras.layers.Dense(100,activation = 'elu',kernel_initializer='he_normal',kernel_constraint = keras.constraints.max_norm(1.))

<Dense name=dense_24, built=False>

# 11.5 总结和实用指南


---

## 一、核心默认配置表（直接照着用）
### 通用DNN默认配置（绝大多数场景首选）
| 超参数 | 默认值 | 说明 |
|--------|--------|------|
| 内核初始化 | He 初始化 | 适配ELU/ReLU等非饱和激活函数，避免梯度消失/爆炸 |
| 激活函数 | ELU | 比ReLU更平滑，缓解神经元死亡，收敛更快 |
| 归一化 | 浅层网络：不需要；深度网络：批量归一化（Batch Normalization） | 深度网络必须加，加速收敛、稳定训练 |
| 正则化 | 提前停止（Early Stopping）；如需更强防过拟合，可加L2正则化 | 提前停止是最安全、最通用的防过拟合手段 |
| 优化器 | 带动量的SGD（或RMSProp、Nadam、Adam） | 带动量的SGD泛化性最好，Adam收敛最快，按需选择 |
| 学习率调度 | 1周期（1Cycle）学习率 | 大幅加速收敛，减少调参成本 |

---

### 自归一化网络（SELU激活）专用配置
 适用场景：仅用于**简单堆叠的密集层网络**，可实现自归一化，无需批量归一化
| 超参数 | 默认值 | 说明 |
|--------|--------|------|
| 内核初始化 | LeCun 初始化 | 专门适配SELU激活，保证初始输出方差稳定 |
| 激活函数 | SELU | 自带自归一化特性，无需批量归一化 |
| 归一化 | 不需要（自归一化） | 网络会自动保持均值0、方差1，无需额外归一化 |
| 正则化 | 如需：Alpha Dropout | 专门适配SELU的Dropout变体，不破坏自归一化特性 |
| 优化器 | 带动量的SGD（或RMSProp、Nadam） | 同通用配置，Adam也可使用 |
| 学习率调度 | 1周期（1Cycle）学习率 | 同通用配置 |

---

## 二、通用前置要求
**所有配置的前提：必须对输入特征做归一化/标准化**
- 这是神经网络训练的基础，不做归一化会导致训练不稳定、收敛极慢
- 常用方法：StandardScaler（标准化）、MinMaxScaler（归一化）

---

## 三、数据场景适配指南
根据你的数据情况，选择不同的训练策略：
1.  **有相似任务的大量标注数据**：直接使用迁移学习，复用预训练模型的特征提取能力
2.  **大量未标记数据、少量标记数据**：使用无监督预训练（如自编码器），再微调
3.  **相似任务的部分神经网络**：直接复用预训练模型的部分层，做迁移学习

---

## 四、特殊场景的定制化方案
### 1. 稀疏模型场景（低内存、高推理速度）
- 方案1：使用**L1正则化**，训练后将绝对值极小的权重置零（诱导权重稀疏）
- 方案2：使用**TensorFlow Model Optimization Toolkit（TF-MOT）**，提供剪枝API，在训练中迭代删除冗余连接
- 注意：L1正则化会破坏自归一化，因此稀疏模型需使用通用DNN配置，而非SELU自归一化配置

### 2. 低延迟模型场景（实时推理、快速预测）
- 核心目标：减少推理时间、降低模型复杂度
- 方案：
  1.  减少网络层数，使用更轻量的结构
  2.  将批量归一化融合到前一层的权重中，减少推理时的计算开销
  3.  替换为更快的激活函数（如Leaky ReLU、仅使用ReLU）
  4.  结合稀疏模型技术，进一步压缩模型
  5.  降低浮点精度：从32位浮点数→16位→8位，大幅提升推理速度
  6.  使用TF-MOT工具包做模型优化

### 3. 风险敏感/高可靠性应用场景
- 核心需求：需要可靠的概率估计、模型不确定性评估
- 方案：使用**MC Dropout（蒙特卡洛Dropout）**
  - 训练时正常使用Dropout，预测时保持Dropout开启，多次采样取平均
  - 同时得到预测结果和模型不确定性，提升决策可靠性
  - 不影响推理延迟的场景下优先使用

---

## 五、关键补充说明
1.  **配置不是硬性规定**：以上默认配置是“开箱即用”的最佳实践，可根据任务特性灵活调整，无需严格遵守
2.  **超参数调整优先级**：先调学习率、批量大小，再调正则化系数、网络结构，最后调优化器超参数
3.  **深度网络训练的核心逻辑**：
    - 通用场景：He初始化+ELU+批量归一化+提前停止+Adam/带动量SGD+1周期学习率
    - 自归一化场景：LeCun初始化+SELU+Alpha Dropout+1周期学习率
    - 特殊场景：根据需求定制，如稀疏模型用L1、低延迟用模型压缩、高可靠用MC Dropout

---

## 六、后续进阶方向
当需要更灵活的控制（如自定义损失函数、自定义训练算法）时，需要使用TensorFlow的底层API，对应本书第12章内容。

---

### 补充：配置速查口诀
- 通用网络：He初+ELU+BN+早停+Adam+1周期
- 自归一化：LeCun+SELU+AlphaDropout+1周期
- 稀疏模型：L1正则+TF-MOT剪枝
- 低延迟：减层+融合BN+低精度+稀疏化
- 高可靠：MC Dropout+不确定性评估

---
